## Numerical Accuracy Benchmark for Triangular Matrix Inverse
 This notebook provides a reusable framework to compare the numerical accuracy of multiple algorithms as a function of input matrix size.

 To use the notebook, the dependencies can be installed in a Python `venv` with:
 ```
 pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cpu
 ```

In [ ]:
import numpy as np
import torch
import time
from typing import Callable, Dict

np.random.seed(0)


def generate_matrix(n: int, kind: str = "gdn", dtype=np.float64, scale: float=0.2) -> np.ndarray:
    """
    Generate test matrices of various types.

    Parameters
    ----------
    n : int
        Matrix dimension (n x n)
    kind : str
        Type of matrix: 'random', 'spd', 'hilbert', 'ill_conditioned', 'gdn', 'true_gdn'
    dtype : numpy dtype
    scale: float
        Controls off-diagonal magnitude. For 'true_gdn', scales beta (gate strength).

    Returns
    -------
    A : ndarray (n x n)
    """
    if kind == "random":
        A = np.triu(scale * np.random.rand(n, n).astype(dtype))
        np.fill_diagonal(A, 1.0)
    elif kind == "random_0_5":
        A = np.triu(0.5 * np.random.rand(n, n).astype(dtype))
        np.fill_diagonal(A, 1.0)
    elif kind == "spd":
        M = np.random.randn(n, n)
        A = M.T @ M + n * np.eye(n)
    elif kind == "hilbert":
        i = np.arange(1, n + 1)
        A = 1.0 / (i[:, None] + i[None, :] - 1.0)
    elif kind == "ill_conditioned":
        U, _ = np.linalg.qr(np.random.randn(n, n))
        V, _ = np.linalg.qr(np.random.randn(n, n))
        s = np.logspace(0, -12, n)
        A = U @ np.diag(s) @ V.T
    elif kind == "delta_net":
        K = np.random.rand(n, n).astype(np.float64)        
        K = K / (1.0001 * np.linalg.norm(K, axis=-1, keepdims=True))
        A = np.triu(K @ K.T, k=1)
        np.fill_diagonal(A, 1.0)
    elif kind == "delta_net_decay":
        K = np.random.rand(n, n).astype(np.float64)        
        K = K / (1.0001 * np.linalg.norm(K, axis=-1, keepdims=True))
        beta = np.diag(np.random.rand(n)).astype(np.float64)
        k_beta = beta @ K
        A = np.triu(k_beta @ K.T, k=1)
        np.fill_diagonal(A, 1.0)
    else:
        raise ValueError(f"Unknown matrix kind: {kind}")

    return A.astype(dtype)


def matmul(A: torch.Tensor, B: torch.Tensor, matmul_dtype: torch.dtype, ret_dtype: torch.dtype):

    A_ = A.to(matmul_dtype)
    B_ = B.to(matmul_dtype)
    C_ = A_ @ B_
    return C_.to(ret_dtype)

def matmul_acc(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, matmul_dtype: torch.dtype, ret_dtype: torch.dtype):

    AB = matmul(A, B, matmul_dtype=matmul_dtype, ret_dtype=matmul_dtype)
    return (C + AB).to(ret_dtype)

### Algorithms under consideration

In [ ]:
import numpy as np
import torch
from math import ceil

def numpy_inv(A: np.ndarray, input_dtype: np.dtype) -> np.ndarray:
    return np.linalg.inv(A.astype(input_dtype))

def reference_inverse(A: torch.Tensor) -> torch.Tensor:
    """High-accuracy reference using NumPy (double precision)."""
    A_inv = numpy_inv(A.numpy(), input_dtype = np.float64) 
    return torch.from_numpy(A_inv)

def tri_inv_vcs(A: torch.Tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype) -> torch.Tensor:
    """
    Compute the inverse of an invertible upper-triangular matrix U
    using the entrywise backward-substitution formula.

    Parameters
    ----------
    A : (n, n) Tensor
        Upper-triangular matrix of size n (with nonzero diagonals).

    Returns
    -------
    A_inv : Square matrix of size n.
        The inverse of A, also upper-triangular.
    """
    A_ = A.to(input_dtype)
    n = A_.shape[0]
    A_inv = torch.zeros_like(A_, dtype=input_dtype)

    # Invert diagonal entries
    for i in range(n):
        A_inv[i, i] = 1.0 / A_[i, i]

    # Compute upper-triangular off-diagonal entries
    for j in range(n):  # column
        # Compute each column backwards (from j - 1 down to 0)
        for i in range(j-1, -1, -1):
            s = 0.0
            for k in range(i+1, j+1):
                s -= (A_[i, k] * A_inv[k, j]) / A_[i, i]
            A_inv[i, j] = s

    return A_inv


def tri_inv_mcs(A: torch.Tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype) -> torch.Tensor:
    "Returns A^{-1} using matmul-based column-sweep substitution."
    A_ = A.to(input_dtype)
    n = A_.shape[0]
    A_inv = torch.zeros_like(A_, dtype=input_dtype)
    I_n = torch.eye(n, dtype=input_dtype)

    A_ = 2 * I_n - A_
    A_inv = torch.eye(n, dtype=input_dtype)
    for k in reversed(range(1, n)):
        M = torch.eye(n, dtype=input_dtype)
        M[:, k] = A_[:, k]
        A_inv = matmul(M, A_inv, matmul_dtype, ret_dtype=input_dtype)
    M = torch.eye(n, dtype=input_dtype)
    M[:, 0] = A_[:, 0]
    A_inv = matmul(M, A_inv, matmul_dtype, ret_dtype=output_dtype)

    return A_inv


def iterative_refinement(A: torch.Tensor, X: torch.Tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype, refinement_steps: int = 1) -> torch.Tensor:
    """Newton iterative refinement of an approximate inverse X of A.

    Dtype contract: returned tensor is always in `output_dtype`.
      - refinement_steps == 0: no refinement, just cast X to output_dtype.
      - refinement_steps == 1: a single step, with ret_dtype=output_dtype.
      - refinement_steps == k >= 2: first k-1 steps use input_dtype for the
        intermediate accumulator; the final step uses output_dtype.
    """
    if refinement_steps == 0:
        return X.to(output_dtype)
    I = torch.eye(A.shape[0], dtype=matmul_dtype)
    A_ = A.to(input_dtype)
    for step in range(refinement_steps):
        ret_dtype = output_dtype if step == refinement_steps - 1 else input_dtype
        R = matmul_acc(-A_, X, I, matmul_dtype, ret_dtype=matmul_dtype)
        X = matmul_acc(X, R, X, matmul_dtype, ret_dtype=ret_dtype)
    return X


def tri_inv_mch(A: torch.Tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype, refinement_steps=0, max_block_size=0) -> torch.Tensor:
    """
    Compute (I + A)^{-1} without explicit inversion
    """
    n = A.shape[0]
    if max_block_size == 0:
        max_block_size = n
    max_iters = int(ceil(np.log2(max_block_size // 2)))

    A_ = A.to(input_dtype)
    I = torch.eye(n, dtype=input_dtype)
    X = 2 * I - A_
    Y = A_ - I
    for _ in range(max_iters-1):
        Y = matmul(Y, Y, matmul_dtype, ret_dtype=matmul_dtype)
        X = matmul_acc(X.to(input_dtype), Y.to(input_dtype), X, matmul_dtype, ret_dtype=matmul_dtype)
    Y = matmul(Y, Y, matmul_dtype, ret_dtype=matmul_dtype)
    X = matmul_acc(X.to(input_dtype), Y.to(input_dtype), X, matmul_dtype, ret_dtype=output_dtype)
    return iterative_refinement(A, X, input_dtype, matmul_dtype, output_dtype, refinement_steps)

def tri_inv_mch_ir(A: torch.Tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype) -> torch.Tensor:
    return tri_inv_mch(A, input_dtype, matmul_dtype, output_dtype, refinement_steps=1)

def tri_inv_mbh(A: torch.tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype, X=None, starting_block_size=1) -> torch.Tensor:
    def even_blocks(A: torch.Tensor, bsz) -> torch.Tensor:
      n = A.shape[0]
      B = torch.zeros((n, n), dtype=A.dtype)
      for idx in range(0, n, int(2 * bsz)):
          B[idx:idx + bsz, idx:idx + bsz] = A[idx:idx + bsz, idx:idx + bsz]
      return B

    def odd_blocks(A: torch.Tensor, bsz) -> torch.Tensor:
      n = A.shape[0]
      B = torch.zeros((n, n), dtype=A.dtype)
      for idx in range(bsz, n, 2 * bsz):
          B[idx:idx + bsz, idx:idx + bsz] = A[idx:idx + bsz, idx:idx + bsz]
      return B

    A_ = A.to(input_dtype)
    n = A_.shape[0]
    I = torch.eye(n, dtype=matmul_dtype)
    MA = -(A_ - I)
    if X is None:
        X = torch.eye(n, dtype=input_dtype)
    block_size = starting_block_size
    while block_size < n/2:
        LX = even_blocks(X, block_size)
        RX = odd_blocks(X, block_size)
        Y = matmul_acc(LX, MA, I, matmul_dtype, ret_dtype=input_dtype)
        X = matmul_acc(Y, RX, LX.to(matmul_dtype), matmul_dtype, ret_dtype=input_dtype)
        # X = (LX @ MA + I) @ RX + LX
        block_size = block_size * 2
    LX = even_blocks(X, int(n/2))
    RX = odd_blocks(X, int(n/2))
    Y = matmul_acc(LX, MA, I, matmul_dtype, ret_dtype=input_dtype)
    X = matmul_acc(Y, RX, LX, matmul_dtype, ret_dtype=output_dtype)     
    return X

def tri_inv_mbh_ir(A: torch.Tensor, input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype) -> torch.Tensor:
    X = tri_inv_mbh(A, input_dtype, matmul_dtype, output_dtype)
    return iterative_refinement(A, X, input_dtype, matmul_dtype, output_dtype, refinement_steps=1)


def tri_inv_mxr(A,  input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype, refinement_steps = 0) -> torch.Tensor:
    def diag_blocks(A: torch.Tensor, bsz) -> torch.Tensor:
      n = A.shape[0]
      B = torch.zeros((n, n), dtype=A.dtype)
      for idx in range(0, n, bsz):
          B[idx:idx + bsz, idx:idx + bsz] = A[idx:idx + bsz, idx:idx + bsz]
      return B
    block_size = 16
    DA = diag_blocks(A, block_size)
    X = tri_inv_mch(DA, input_dtype, matmul_dtype, output_dtype, refinement_steps=refinement_steps, max_block_size=block_size)
    X = tri_inv_mbh(A, input_dtype, matmul_dtype, output_dtype, X, starting_block_size=block_size)
    return X

def tri_inv_mxr_ir(A,  input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype):
    return tri_inv_mxr(A, input_dtype, matmul_dtype, output_dtype, refinement_steps=1)

def tri_inv_ns(A,  input_dtype: torch.dtype, matmul_dtype: torch.dtype, output_dtype: torch.dtype, num_iters: int = 0):
    n = A.shape[-1]
    if num_iters == 0:
      num_iters = ceil(2 * np.log2(n)) + 2 # TODO(anastasios): Can num_iters be smaller than 2log(n)? 1.5x fails!
    I = torch.eye(n, dtype=input_dtype)

    # scale matrix for stability
    scale = 1/(2*n) 
    X = I * scale
    A_ = A.to(input_dtype) 


    for _ in range(num_iters-1):
        Y = matmul(A_, X, matmul_dtype, ret_dtype=input_dtype)
        # 2*I : can be implemented as 2 matmuls: (1) C = I @ I, (2) C += I @ I
        X = matmul(X, 2*I - Y, matmul_dtype, ret_dtype=input_dtype)

    Y = matmul(A_, X, matmul_dtype, ret_dtype=input_dtype)
    X = matmul(X, 2*I - Y, matmul_dtype, ret_dtype=output_dtype)

    return X

ALGORITHMS: Dict[str, Callable[[torch.Tensor, torch.dtype, torch.dtype, torch.dtype], torch.Tensor]] = {
    # "np_inv(fp32)": algo_numpy_inv,
    "VCS": tri_inv_vcs,
    "MCS": tri_inv_mcs,
    "MCH": tri_inv_mch,
    "MBH": tri_inv_mbh,
    "MXR": tri_inv_mxr,
    "MCH+IR": tri_inv_mch_ir,
    "MBH+IR": tri_inv_mbh_ir,
    "MXR+IR": tri_inv_mxr_ir,
    "NS": tri_inv_ns,
    # "Newton-Schulz+IR": tri_inv_ns_ir,
    # "Custom": algo_custom
}


### Accuracy metrics (forward/backward error)

In [ ]:
import numpy as np
from numpy.linalg import norm

def max_abs_error(X_hat: torch.Tensor, X_ref: torch.Tensor) -> float:
    """max(abs(X_hat - X_ref))"""
    return np.max(np.abs(X_hat.numpy() - X_ref.numpy()))

def max_rel_error(X_hat: torch.Tensor, X_ref: torch.Tensor) -> float:
    """max(abs(X_hat - X_ref) / abs(X_ref))"""
    max_error = 0
    X_hat_np = X_hat.numpy()
    X_ref_np = X_ref.numpy()
    n = X_hat_np.shape[0]
    for i in range(n):
        for j in range(n):
            if X_ref_np[i,j] == 0:
                continue
            err_ij = np.abs(X_hat_np[i,j] - X_ref_np[i,j]) / np.abs(X_ref_np[i,j])
            max_error = max_error if err_ij < max_error else err_ij
    return max_error

def frobenius_rel_error(X_hat: torch.Tensor, X_ref: torch.Tensor) -> float:
    """||X_hat - X_ref||_F / ||X_ref||_F"""
    X_hat_np = X_hat.numpy()
    X_ref_np = X_ref.numpy()
    return norm(X_hat_np - X_ref_np, ord='fro') / norm(X_ref_np, ord='fro')


def backward_abs_error(A: torch.Tensor, X_hat: torch.Tensor) -> float:
    """||I - A X_hat||_F"""
    n = A.shape[0]
    X_hat_np = X_hat.numpy()
    I = np.eye(n, dtype=X_hat_np.dtype)
    return norm(I - A.numpy() @ X_hat_np, ord='fro')

### Benchmark runner

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Dtypes:
    input_dtype: torch.dtype
    input_dtype_np: np.dtype
    matmul_dtype: torch.dtype
    output_dtype: torch.dtype

@dataclass
class BenchmarkConfig:
    sizes: List[int]
    dtypes: List[Dtypes]
    matrix_kind: str = "random"
    matrix_scale: float = 1
    repeats: int = 3

@dataclass
class Result:
    size: int
    algo: str
    input_dtype: str
    matmul_dtype: str
    output_dtype: str
    frobenius_rel_err: float
    backward_abs_err: float
    max_abs_err: float
    max_rel_err: float
    time_sec: float


def run_benchmark(config: BenchmarkConfig) -> List[Result]:
    results: List[Result] = []
    ref_dtype = torch.float64
    for n in config.sizes:
        A_np = generate_matrix(n, kind=config.matrix_kind, dtype=np.float64, scale=config.matrix_scale)
        A_torch = torch.from_numpy(A_np)
        # reference always computed in float64 for stability
        X_ref = reference_inverse(A_torch)
        for dtypes in config.dtypes:
            A = A_torch.to(dtypes.input_dtype)

            for algo_name, algo_fn in ALGORITHMS.items():
                frobenius_rel_errs = []
                backward_abs_errs = []
                max_abs_errs = []
                max_rel_errs = []
                times = []

                for _ in range(config.repeats):
                    start = time.perf_counter()
                    X_hat = algo_fn(A, dtypes.input_dtype, dtypes.matmul_dtype, dtypes.output_dtype)
                    elapsed = time.perf_counter() - start
                    X_hat_in_ref_dtype = X_hat.to(ref_dtype)

                    frobenius_rel_errs.append(frobenius_rel_error(X_hat_in_ref_dtype, X_ref))
                    backward_abs_errs.append(backward_abs_error(A_torch, X_hat_in_ref_dtype))
                    max_abs_errs.append(max_abs_error(X_hat_in_ref_dtype, X_ref))
                    max_rel_errs.append(max_rel_error(X_hat_in_ref_dtype, X_ref))
                    times.append(elapsed)

                results.append(Result(
                    size=n,
                    algo=algo_name,
                    input_dtype=str(dtypes.input_dtype),
                    matmul_dtype=str(dtypes.matmul_dtype),
                    output_dtype=str(dtypes.output_dtype),
                    frobenius_rel_err=float(np.mean(frobenius_rel_errs)),
                    backward_abs_err=float(np.mean(backward_abs_errs)),
                    max_abs_err=float(np.mean(max_abs_errs)),
                    max_rel_err=float(np.mean(max_rel_errs)),
                    time_sec=float(np.mean(times)),
                ))

    return results

## Run Benchmark


In [ ]:
config = BenchmarkConfig(
    sizes=[16, 32, 64, 128], 
    dtypes=[Dtypes(torch.float32,np.float32,torch.float32,torch.float32), 
            Dtypes(torch.float16,np.float16,torch.float32,torch.float32),
            # Dtypes(torch.float16,np.float16,torch.float16,torch.float16),
            Dtypes(torch.bfloat16,np.float32,torch.float32,torch.float32),
            # Dtypes(torch.bfloat16,np.float32,torch.bfloat16,torch.bfloat16)
            ],
    matrix_kind="delta_net",
    matrix_scale=1.0,
    repeats=3,
)

results = run_benchmark(config)

### Process results

In [ ]:
import pandas as pd

df = pd.DataFrame([r.__dict__ for r in results])
df.head(25)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Seaborn style
sns.set_theme(style="whitegrid", font_scale=2)
sns.set_context(rc = {'patch.linewidth': 0.1})

# Ensure consistent algorithm order
algo_order = ALGORITHMS.keys()

n_benches = 3

fig, ax = plt.subplots(nrows=3, ncols=n_benches, figsize=(18, 10), sharex=True, sharey=True)

COLOR_PALETTE = ["#d9f0d3", "#7fbf7b", "#1b7837", "#1B1B1B" ]

plot_anchors = []
for i, dtypes in enumerate(config.dtypes):
    ax[0, i].set_title(f"{str(dtypes.input_dtype)}".replace("torch.",""))
    plot_df = df[(df["input_dtype"] == str(dtypes.input_dtype)) & (df["matmul_dtype"] == str(dtypes.matmul_dtype))].copy()
    for j, err_measure in enumerate(["max_abs_err", "max_rel_err", "frobenius_rel_err"]):
        plot_anchors.append(
            sns.barplot(
                data=plot_df,
                x="algo",
                y=err_measure,
                hue="size",              # grouped bars by matrix size
                order=algo_order,
                estimator="mean",        # or "median" (often better for errors)
                errorbar=None,             # disable CI bars (paper-friendly)
                palette=COLOR_PALETTE,
                width=0.8,
                ax=ax[j,i]
            )
        )
        plot_anchors[-1].legend_.remove()
        plot_anchors[-1].set_yscale('log')
        plot_anchors[-1].set_ylim(1e-8, 1e+3)
        plot_anchors[-1].set_yticks([1e-8, 1e-6, 1e-4, 1e-2, 1, 1e+2, 1e+4])
        for bar in plot_anchors[-1].patches:
            bar.set_linewidth(1.5)
            bar.set_edgecolor('black')
    plot_anchors[-1].tick_params(axis='x', rotation=60)

ax[0,0].set_ylabel("Max abs. err.")
ax[1,0].set_ylabel("Max rel. err.")
ax[2,0].set_ylabel("Frob. rel. err.")
for i in range(n_benches):
    ax[2,i].set_xlabel("Algorithm")
handles, labels = ax[0,1].get_legend_handles_labels()
fig.legend(
    handles=handles,
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.55, 1.05),
    framealpha=0.9,
    markerscale=1.2,
)

plt.tight_layout()
plt.show()


### Save results to output

In [ ]:
# Save results to CSV for paper-quality plots
output_csv = "numerical_accuracy_results.csv"
df.to_csv(output_csv, index=False)
print(f"Results saved to {output_csv}")